<a href="https://colab.research.google.com/github/FadhilTaufiqurrachman/Machine-Learning/blob/main/244107020090_Fadhil_Taufiqurrachman_T11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Praktikum Jobsheet 11

NIM : 244107020090

Nama : Fadhil Taufiqurrachman

Kelas : TI 3G

No Absen : 06

Mata Kuliah : Pembelajaran Mesin

Persiapan Data

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Education/Current Semester College File /Machine Learning/'

Mounted at /content/drive


Import Library

In [ ]:
# Import seluruh library yang dibutuhkan untuk Nomor 1 dan 2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

### Dataset voice.csv

In [ ]:
# Memuat dataset voice.csv
df_voice = pd.read_csv(BASE_PATH + 'voice.csv')

# Memisahkan fitur (X) dan label target (y)
X_voice = df_voice.drop('label', axis=1)
y_voice = df_voice['label']

# Melakukan encoding pada label teks ('male' dan 'female' menjadi 1 dan 0)
encoder = LabelEncoder()
y_voice_encoded = encoder.fit_transform(y_voice)

# Melakukan standarisasi data (Penting untuk algoritma SVM agar skala fitur seragam)
scaler = StandardScaler()
X_voice_scaled = scaler.fit_transform(X_voice)

display(df_voice.head())

,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


In [ ]:
# Split data rasio 70:30
X_train_70, X_test_30, y_train_70, y_test_30 = train_test_split(
    X_voice_scaled, y_voice_encoded, test_size=0.3, random_state=42)

# Split data rasio 80:20
X_train_80, X_test_20, y_train_80, y_test_20 = train_test_split(
    X_voice_scaled, y_voice_encoded, test_size=0.2, random_state=42)

In [ ]:
kernels = ['linear', 'poly', 'rbf']
results = []

# Iterasi untuk menguji setiap jenis kernel pada masing-masing rasio split
for kernel in kernels:
    model = SVC(kernel=kernel)

    # Pelatihan dan Evaluasi untuk Split 70:30
    model.fit(X_train_70, y_train_70)
    acc_70 = accuracy_score(y_test_30, model.predict(X_test_30))

    # Pelatihan dan Evaluasi untuk Split 80:20
    model.fit(X_train_80, y_train_80)
    acc_80 = accuracy_score(y_test_20, model.predict(X_test_20))

    results.append([kernel, acc_70, acc_80])

# Tabulasi performansi metrik akurasi
df_results = pd.DataFrame(results, columns=['Kernel', 'Akurasi (70:30)', 'Akurasi (80:20)'])
display(df_results)

,Kernel,Akurasi (70:30),Akurasi (80:20)
0,linear,0.970557,0.976341
1,poly,0.956887,0.968454
2,rbf,0.981073,0.982650


Eksperimen pada dataset voice menunjukkan secara empiris bagaimana variasi porsi data latih dan pemilihan trik kernel memengaruhi kemampuan generalisasi Support Vector Machine. Peningkatan rasio data latih menjadi 80:20 secara umum memberikan sedikit margin peningkatan pada skor akurasi karena model memiliki lebih banyak sampel untuk memetakan batas keputusan (decision boundary) dengan presisi. Di antara ketiga parameter, kernel Linear dan RBF biasanya menghasilkan performansi terbaik dan nyaris identik (seringkali menyentuh akurasi >97% setelah fitur distandarisasi), yang mengindikasikan bahwa varians parameter akustik suara pria dan wanita dalam dataset ini sejatinya memiliki pemisahan fitur (hyperplane) yang cukup linier dan sangat terstruktur, sedangkan kernel polinomial cenderung bekerja kurang efisien tanpa adanya penyesuaian degree yang spesifik

### Dataset Citra

In [ ]:
# Fungsi untuk mengekstrak histogram warna 3D (Hue, Saturation, Value)
def extract_histogram(image):
    # Mengubah format warna dari RGB (plot default) ke HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    # Membuat histogram menggunakan 8 bins untuk setiap channel (H, S, V)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256])

    # Normalisasi array dan meratakannya menjadi array 1 Dimensi (512 fitur)
    cv2.normalize(hist, hist)
    return hist.flatten()

# Fungsi untuk memuat seluruh gambar siang dan malam lalu mengekstraknya
def load_and_extract_hist(img_dirs):
    features = []
    labels = []

    for img_dir in img_dirs:
        p = Path(img_dir)
        for label_name in ['day', 'night']:
            dir_path = p / label_name
            if dir_path.exists():
                for file in dir_path.glob('*.jpg'):
                    # Membaca gambar
                    img = plt.imread(file)
                    if img is not None:
                        # Resize gambar agar komputasi pembuatan histogram lebih ringan dan seragam
                        img_resized = cv2.resize(img, (300, 300))

                        # Ekstrak histogram dan simpan ke list
                        hist_feat = extract_histogram(img_resized)
                        features.append(hist_feat)

                        # Label: day = 1, night = 0
                        labels.append(1 if label_name == 'day' else 0)

    return np.array(features), np.array(labels)

In [ ]:
# Menggabungkan seluruh data citra dari folder training dan testing praktikum sebelumnya
img_folders = [BASE_PATH + "images/training/", BASE_PATH + "images/test/"]

print("Mengekstrak fitur histogram dari citra, harap tunggu...")
X_img, y_img = load_and_extract_hist(img_folders)
print(f"Total data gambar berhasil diekstrak: {len(X_img)} gambar.")

# Split data menggunakan rasio 80:20 sesuai instruksi
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_img, y_img, test_size=0.2, random_state=42)

Mengekstrak fitur histogram dari citra, harap tunggu...
Total data gambar berhasil diekstrak: 400 gambar.


In [ ]:
# Menentukan kombinasi nilai tuning parameter C dan gamma untuk dieksplorasi
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1]
}

# Menggunakan GridSearchCV untuk mencari model RBF paling optimal
svm_rbf = SVC(kernel='rbf')
grid_search = GridSearchCV(svm_rbf, param_grid, cv=5, n_jobs=-1)

print("Memulai pencarian hyperparameter terbaik (Tuning)...")
grid_search.fit(X_train_img, y_train_img)

# Menampilkan parameter pemenang
print("Hyperparameter Terbaik yang ditemukan:", grid_search.best_params_)

# Mengukur dan mencatat performansi akurasi pada data testing
best_model = grid_search.best_estimator_
acc_img_test = accuracy_score(y_test_img, best_model.predict(X_test_img))

print(f"Akurasi Final Model (SVM-RBF Histogram): {acc_img_test * 100:.2f}%")

Memulai pencarian hyperparameter terbaik (Tuning)...
Hyperparameter Terbaik yang ditemukan: {'C': 100, 'gamma': 0.1}
Akurasi Final Model (SVM-RBF Histogram): 98.75%


Penggantian metrik sederhana "average brightness" dengan fitur ekstraksi Color Histogram pada klasifikasi citra secara radikal mengubah dimensi pembelajaran algoritma dari variabel tunggal (1D) menjadi vektor ruang warna yang kaya dan kompleks (berisi 512 fitur distribusi warna HSV). Ekstraksi ini memungkinkan model untuk tidak hanya menebak dari "seberapa terang" gambar, melainkan mempelajari dominasi pola warna spesifik (seperti siluet jingga saat sore atau rona lampu kuning malam hari). Karena kompleksitas vektor ini bersifat non-linier, penggunaan algoritma SVM kernel RBF dipadukan dengan teknik GridSearchCV menjadi sangat krusial; metode tuning ini sukses menavigasi multidimensi tersebut untuk menyeimbangkan nilai penalti (C) dan fleksibilitas garis kurva (gamma), sehingga mencegah model mengalami overfitting sekaligus mendongkrak performa akurasi hingga di atas pencapaian pengujian pada praktikum sebelumnya.